# PHIL-TEXT — Faz 3.4: Feature Engineering

**Amaç:** Temizlenmiş metinlerden modele beslenecek özellik vektörleri türetmek.

### Adımlar
1. Temiz corpus yükle
2. İstatistiksel özellikler (17 adet)
3. Özellik dağılımları & korelasyon analizi
4. Philosopher bazında özellik profili
5. Label Encoding
6. TF-IDF Özellik Matrisi
7. TF-IDF görselleştirme
8. Chunk tabanlı split (train/val/test)
9. Özellikleri kaydet

## 1. Kurulum & Yükleme

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.sparse import issparse

from src.data.load_data import chunk_texts
from src.features.build_features import (
    compute_statistical_features, encode_labels, build_tfidf_features,
    split_dataset, scale_features, save_features, save_stat_features,
    STAT_FEATURE_COLS,
)

plt.rcParams['figure.dpi'] = 130
sns.set_theme(style='whitegrid', palette='muted')

PHIL_COLORS = {
    'platon': '#4C72B0', 'aristoteles': '#DD8452', 'marcus_aurelius': '#55A868',
    'descartes': '#C44E52', 'spinoza': '#8172B2', 'locke': '#937860',
    'hume': '#DA8BC3', 'kant': '#8C8C8C', 'schopenhauer': '#CCB974', 'nietzsche': '#64B5CD',
}

# Temizlenmiş corpus
df_clean = pd.read_parquet('../data/processed/corpus_clean.parquet')
print(f'Temiz corpus: {df_clean.shape[0]} eser')
df_clean[['philosopher','work','word_count']].head()

## 2. İstatistiksel Özellikler

In [ ]:
df_feat = compute_statistical_features(df_clean, text_col='text')

stat_cols = [c for c in STAT_FEATURE_COLS if c in df_feat.columns]
print(f'Toplam istatistiksel ozellik: {len(stat_cols)}')
print(df_feat[stat_cols].describe().round(4))

In [ ]:
print('=== Ozellik Tablosu (eser bazli) ===')
display_cols = ['philosopher','work','avg_word_len','avg_sent_len','unique_word_ratio',
                'stopword_ratio','punct_density','hapax_ratio','vocab_richness']
df_feat[display_cols].sort_values('vocab_richness', ascending=False).reset_index(drop=True)

## 3. Özellik Dağılımları & Korelasyon

In [ ]:
key_feats = ['avg_word_len','avg_sent_len','unique_word_ratio',
             'stopword_ratio','punct_density','hapax_ratio','vocab_richness','long_word_ratio']
existing = [f for f in key_feats if f in df_feat.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(existing):
    ax = axes[i]
    for phil, grp in df_feat.groupby('philosopher'):
        ax.scatter([phil] * len(grp), grp[feat],
                   color=PHIL_COLORS.get(phil, '#888'), s=80, alpha=0.85, zorder=3)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

plt.suptitle('Filozofa Göre Özellik Dağılımları', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/feat_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Korelasyon matrisi
corr_cols = [c for c in STAT_FEATURE_COLS if c in df_feat.columns]
corr = df_feat[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', ax=ax,
            mask=mask, square=True, linewidths=0.3, vmin=-1, vmax=1,
            annot_kws={'size': 7})
ax.set_title('İstatistiksel Özellikler Korelasyon Matrisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/feat_correlation.png', bbox_inches='tight')
plt.show()

## 4. Philosopher Bazında Özellik Profili (Radar)

In [ ]:
# Normalize edilmiş özellik profili (min-max) — tablo
profile_feats = ['avg_word_len','avg_sent_len','unique_word_ratio',
                 'punct_density','hapax_ratio','vocab_richness']
profile_feats = [f for f in profile_feats if f in df_feat.columns]

phil_profile = df_feat.groupby('philosopher')[profile_feats].mean()

# Min-max normalize
phil_norm = (phil_profile - phil_profile.min()) / (phil_profile.max() - phil_profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(phil_norm.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(profile_feats)))
ax.set_xticklabels(profile_feats, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(phil_norm)))
ax.set_yticklabels(phil_norm.index, fontsize=9)
plt.colorbar(im, ax=ax, label='Normalize Deger (0=Min, 1=Max)')

for i in range(len(phil_norm)):
    for j in range(len(profile_feats)):
        ax.text(j, i, f'{phil_norm.values[i,j]:.2f}', ha='center', va='center', fontsize=7)

ax.set_title('Filozof Bazında Özellik Profili (Min-Max Normalize)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/feat_profile_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Label Encoding

In [ ]:
result = encode_labels(df_feat)
df_encoded = result['df']
mappings = result['mappings']

print('=== Label Mapping ===')
for col, m in mappings.items():
    print(f'  {col}: {m}')
print()
df_encoded[['philosopher','philosopher_id','era','era_id','school','school_id']].drop_duplicates().sort_values('philosopher_id')

## 6. TF-IDF Özellik Matrisi

In [ ]:
# Chunk üzerinden TF-IDF (chunk = model girişi)
df_chunks = chunk_texts(df_encoded, chunk_size=512, overlap=64)

# Chunk'lara da label ekle
label_map = df_encoded.set_index('philosopher')['philosopher_id'].to_dict()
df_chunks['philosopher_id'] = df_chunks['philosopher'].map(label_map)

print(f'Toplam chunk: {len(df_chunks):,}')
print(f'Sinif dagilimi:')
print(df_chunks['philosopher'].value_counts().to_string())

In [ ]:
X_tfidf, tfidf_vec = build_tfidf_features(
    df_chunks['text'].tolist(),
    max_features=10_000,
    ngram_range=(1, 2),
)
print(f'TF-IDF matrisi: {X_tfidf.shape}')
print(f'Bellek: {X_tfidf.data.nbytes / 1024 / 1024:.1f} MB (sparse)')

## 7. TF-IDF Görselleştirme

In [ ]:
# Filozofa göre en ayırt edici terimler (ortalama TF-IDF)
import numpy as np
feature_names = tfidf_vec.get_feature_names_out()

philosophers = sorted(df_chunks['philosopher'].unique())
ncols = 5
nrows = (len(philosophers) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 4))
axes = axes.flatten()

for idx, phil in enumerate(philosophers):
    mask = df_chunks['philosopher'] == phil
    X_phil = X_tfidf[mask.values]
    mean_tfidf = np.asarray(X_phil.mean(axis=0)).flatten()
    top10_idx = mean_tfidf.argsort()[-10:][::-1]
    terms = [feature_names[i] for i in top10_idx]
    scores = mean_tfidf[top10_idx]
    ax = axes[idx]
    ax.barh(list(reversed(terms)), list(reversed(scores)),
            color=PHIL_COLORS.get(phil, '#888'), alpha=0.85)
    ax.set_title(phil.replace('_',' ').title(), fontsize=11, fontweight='bold')
    ax.tick_params(labelsize=8)

for j in range(len(philosophers), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Filozofa Göre En Ayırt Edici TF-IDF Terimleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/feat_tfidf_per_philosopher.png', bbox_inches='tight')
plt.show()

## 8. Train / Val / Test Bölme

In [ ]:
splits = split_dataset(
    df_chunks,
    text_col='text',
    label_col='philosopher_id',
    test_size=0.15,
    val_size=0.15,
)

print('=== Veri Bölme Sonuçları ===')
total = len(df_chunks)
for split_name in ['train', 'val', 'test']:
    y = splits[f'y_{split_name}']
    pct = len(y) / total * 100
    print(f'  {split_name:<5}: {len(y):>4,} örnek ({pct:.1f}%)')

In [ ]:
# Sınıf dağılımını kontrol et
import collections
id2phil = {v: k for k, v in mappings['philosopher'].items()}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, split_name in zip(axes, ['train', 'val', 'test']):
    y = splits[f'y_{split_name}']
    counts = collections.Counter(y)
    phils = [id2phil.get(k, str(k)) for k in sorted(counts)]
    vals  = [counts[k] for k in sorted(counts)]
    ax.bar(phils, vals,
           color=[PHIL_COLORS.get(p,'#888') for p in phils], alpha=0.85)
    ax.set_title(f'{split_name.title()} Sinif Dagilimi', fontsize=11)
    ax.set_xticklabels(phils, rotation=45, ha='right', fontsize=8)

plt.suptitle('Train/Val/Test Sınıf Dağılımı', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/feat_split_distribution.png', bbox_inches='tight')
plt.show()

## 9. Özellikleri Kaydet

In [ ]:
import joblib
import os
os.makedirs('../data/processed', exist_ok=True)

# 1) Text split'leri kaydet
save_features(splits, mappings, output_dir='../data/processed')

# 2) İstatistiksel özellik tablosu
save_stat_features(df_encoded, output_dir='../data/processed')

# 3) Chunk DataFrame (philosopher_id dahil)
df_chunks.to_parquet('../data/processed/corpus_chunks_labeled.parquet', index=False)

# 4) TF-IDF vectorizer
joblib.dump(tfidf_vec, '../models/tfidf_vectorizer.joblib')

print('Kaydedilen dosyalar:')
print('  data/processed/splits.npz')
print('  data/processed/label_mappings.json')
print('  data/processed/corpus_features.parquet')
print('  data/processed/corpus_chunks_labeled.parquet')
print('  models/tfidf_vectorizer.joblib')

In [ ]:
print('=' * 60)
print('         FAZ 3.4 OZET')
print('=' * 60)
print(f"""
  Istatistiksel ozellik  : {len([c for c in STAT_FEATURE_COLS if c in df_feat.columns])}
  TF-IDF ozellik         : {X_tfidf.shape[1]:,} (unigram + bigram)
  Chunk toplam           : {len(df_chunks):,}
  Train / Val / Test     : {len(splits['X_train'])} / {len(splits['X_val'])} / {len(splits['X_test'])}
  Sinif sayisi           : {len(mappings['philosopher'])}
  Label mappings         : {list(mappings['philosopher'].keys())}
  Sonraki adim           : Faz 4.1 — Baseline Modeli
""")
print('=' * 60)